### What this measures

The retrieval evals ask what litesearch *finds*. This one asks what it costs to put anything in:
parsing, chunking, embedding, SQLite upserts, ANN index maintenance, graph extraction. It exists
because a corpus of a million documents fails on the write side long before it fails on the read
side, and nothing in the retrieval evals would have caught it.

Everything here is driven by `evals/ingest_bench.py` and runs on the same `evals.corpus` genres the
retrieval evals use, repartitioned into as many documents as a run needs. `hash_embed` stands in for
an encoder so the numbers are litesearch's own overhead rather than ONNX inference — pass
`--encoder potion` to put a real model back in.

**Read the exponent, not the throughput.** Per-document cost is only a cost if it is constant. Every
size sweep below reports the slope of log(time) on log(n): 1.0 is linear, and anything approaching
2.0 does not reach a million documents at any hardware budget. The single most important number in
this notebook is that `add_dir` went from 1.72 to 0.91.

Cells are `eval: false` because a full sweep takes tens of minutes. The recorded output of each is
in the markdown beneath it, measured on 4 CPUs with SQLite in WAL mode.


In [ ]:
#| eval: false
# everything in this notebook, from the command line
!python -m evals.ingest_bench --help

### The headline

| | before 0.1.16 | after | |
|---|---|---|---|
| `add_dir`, 400 markdown docs | 112.2s, exponent 1.72 | **9.2s, exponent 0.91** | 12.2x |
| code ingestion, 2,169 files | 190.4s | **7.3s** | 26x |
| `process_content` | 469–527 rows/s | **912 rows/s** | 1.8x |
| `resolve_entities`, 8,487 entities | 13.3s, exponent 1.35 | **5.8s, exponent 1.08** | 2.3x |

The multipliers are the least interesting part. The exponents are the finding: ingest used to get
slower per document as the corpus grew, and now it does not.


### 1. Document ingestion, swept by corpus size

`add_doc` used to end with `rebuild_index()`, which reads *every* embedding blob in the store and
reconstructs the whole HNSW graph. Per document that is O(corpus), so a directory walk was O(N²) —
to build an index nobody reads until ingestion finishes.

It now mirrors only that document's keys into the index, the way `sync` always has, and `add_dir`
treats the walk as a bulk load: FTS triggers suspended for the duration, one index rebuild at the
end.


In [ ]:
#| eval: false
from evals.ingest_bench import bench_docs
bench_docs(sizes=(25,50,100,200,400))

```
before (0.1.15)                          after (0.1.16)
add_dir(25)     1.64s   65.5 ms/doc      add_dir(25)    0.80s   32.1 ms/doc
add_dir(50)     3.31s   66.3 ms/doc      add_dir(50)    1.21s   24.1 ms/doc
add_dir(100)    8.35s   83.5 ms/doc      add_dir(100)   2.16s   21.6 ms/doc
add_dir(200)   29.05s  145.2 ms/doc      add_dir(200)   5.06s   25.3 ms/doc
add_dir(400)  112.18s  267.8 ms/doc      add_dir(400)   9.21s   23.0 ms/doc
exponent        1.72                     exponent       0.91
```

Per-document cost used to climb 4x from 25 to 400 documents. It is now flat.

The profile of the old 200-document run is where this came from: `usearch.compiled.add_many` was
17.4s of 34.7s — half the wall clock — alongside 409,730 row fetches and 407,928 `np.frombuffer`
calls, all of them the quadratic re-reads.


In [ ]:
#| eval: false
# the isolating experiment: same corpus, same final index, rebuild deferred to once per batch.
# run against 0.1.15 this reports the gap the fix closed.
from evals.ingest_bench import bench_deferred_index
bench_deferred_index(sizes=(50,100,200,400))

On 0.1.15 this was 1.35x at 50 documents rising to 7.01x at 400 — the gap grows linearly with
corpus size, which is what "unbounded" means here. On 0.1.16 both arms are the fixed path, so it
now measures only the cost of the final rebuild (0.49s of 16.0s at 400 documents).

**Benchmark on an idle machine.** A concurrent `nbdev-test` (2 workers on 4 CPUs) was enough to
turn the 50-document case into 55.9s against 3.7s, while leaving every vector count identical.


### 2. Writes: one transaction, and the FTS triggers

`process_content` called `insert_all` outside any transaction, so apswutils committed per chunk and
WAL fsynced each time. The batching already existed — but behind `parallel=True`, which is a
concurrency flag, not a throughput one. It now always batches; `parallel` controls only the widened
busy timeout, which is all it ever meant.

The remaining gap between a store with FTS and one without is the per-row trigger. `bulk_load`
suspends the triggers and uses FTS5's own `rebuild` once at the end.


In [ ]:
#| eval: false
from evals.ingest_bench import bench_store, verify_bulk_load
bench_store(sizes=(2000,8000,32000))
verify_bulk_load()

```
8,000 rows                            0.1.15    0.1.16
no fts, autocommit                     4.56s
no fts, one txn                        0.50s              9.1x
fts, autocommit                        7.44s
fts, one txn                           1.46s              5.1x
process_content(parallel=True)         1.11s              the win, gated behind a flag

process_content end to end            527/s     913/s     exponent 1.04 -> 1.00
```

`bulk_load` is 2.05x, and populates the index rather than skipping it:

```
bulk_load=False  1.15s  rows=8000  fts_rows=8000  hits(vessel)=37
bulk_load=True   0.66s  rows=8000  fts_rows=8000  hits(vessel)=37
```

Only worth it for a *load*: `rebuild` is O(rows in the table), so wrapping a two-row update in it is
strictly slower than letting the triggers run.


### 3. Code files

Two independent problems, both worth more than they look.

`ast.get_source_segment` re-splits the entire source file on every call, and `pyparse` called it
once per chunk it emitted — O(top-level defs x file size) per file. Profiling 400 files of
`transformers`, `ast._splitlines_no_ff` was **47.3s of 70.0s**. `pyparse` also walked the whole tree
tagging every node with a `parent`, then filtered on "parent is the Module" — which is what
`tree.body` already means.

And `dir2chunks`/`pkg2chunks` ran `file_parse` on a *thread* pool. It is `ast.parse` plus a
pure-python walk and holds the GIL end to end, so the pool contended rather than overlapped.


In [ ]:
#| eval: false
# byte-for-byte against the implementation this replaced, over a real package
from evals.ingest_bench import verify_pyparse
verify_pyparse('litesearch')                  # or any large tree: site-packages/transformers

Over 2,169 files of `transformers`:

```
                                    old        new           chunks        mismatches
default                          159.80s     22.36s   7.1x   14,444 both        0
imports=True, assigns=True       281.89s     24.66s  11.4x   32,720 both        0
```

Zero mismatches on content *and* metadata. `nbs/02_data.ipynb` pins `_seg` against
`ast.get_source_segment` on the cases where the two line-splitters actually differ: a decorated def,
a one-line class, a form feed, and a non-ascii line before the node. An earlier draft that used
`str.splitlines` had 9 mismatches for exactly that reason — form feeds are breaks for `splitlines`
and deliberately not for `_splitlines_no_ff`.

The pool, same 2,169 files:

```
serial (n_workers=0)      17.67s
threads (the old default) 24.00s      0.76x — slower than serial
processes (new default)    7.34s      3.3x
```

190.4s on the old default against 7.34s: **26x** end to end. Directories below
`MIN_PARALLEL_FILES` (64) stay serial, because below that a pool costs more to start than the parse
it saves.


### 4. PDFs

`pdf_parse` is Rust (pdf-oxide) but holds the GIL, so it threads no better than the AST parser does.


In [ ]:
#| eval: false
from evals.ingest_bench import bench_pdf
bench_pdf()

```
8 corpus PDFs, 489 pages
serial         4.07s   120.2 pages/s
thread(4)      4.41s   111.0 pages/s   0.92x
process(4)     2.14s   228.3 pages/s   1.90x
thread(8)      3.92s   124.8 pages/s   1.04x
process(8)     2.68s   182.7 pages/s   1.52x
```

Single-document rates for reference: 120 pages/s on the text-heavy directives, 34 pages/s on an
image-heavy arXiv paper.

**Still open:** `add_dir` walks files serially. `_parse_files` shows the shape of the fix but it has
not been applied to the document path, because PDF parsing and SQLite writing want different pool
sizes.


### 5. Graph

`build_graph` is linear (exponent ~1.0) and simply slow: ~40 chunks/s, so a million chunks is about
7 hours on one core. It also accumulates `ents`, `mens`, `edges` and `wins` in memory for the whole
call, so it cannot be handed a million chunks in one go regardless of speed. Feed it in batches.

`resolve_entities` was superlinear at exponent 1.35. `_lexical_pairs` was the obvious suspect and
the profile said otherwise: of 23.0s, **10.3s was `ann_search`** — one HNSW probe *plus one
`rowid IN (...)` query* per entity, 8,487 of each — and 4.2s was `_toks`, called 252,813 times on a
few thousand strings because the lexical guard re-tokenises both sides of every candidate pair.

So the blocking strategy is untouched and the candidate set is unchanged: one batched probe, and an
`lru_cache` on `_toks`.


In [ ]:
#| eval: false
from evals.ingest_bench import bench_graph
bench_graph(sizes=(500,1000,2000))

```
                        before    after
resolve n=500            3.61s    2.01s
resolve n=1000           7.43s    3.52s
resolve n=2000          13.26s    5.65s    2.3x
exponent                  1.35     1.08
```


#### An honest note on determinism

`resolve_entities` does not return the same answer twice. Rebuilt from scratch at n=2000 it gives
`merged=5896` on one run and `merged=5891` on the next — about 0.1%.

This is not the batching. usearch builds its HNSW graph across threads, so two rebuilds of the same
vectors are two slightly different graphs, and an approximate probe over a different graph returns
slightly different neighbours. It was true before this change too. The check that separates the two
holds the index still and varies only the probe:


In [ ]:
#| eval: false
from evals.ingest_bench import verify_ann_probe, verify_resolve
verify_ann_probe(n=2000)          # one fixed index, batched vs the per-entity loop
verify_resolve(sizes=(1000,2000), reps=2)   # full rebuilds: shows the ~0.1% drift

```
== ann probe equivalence (8487 entities, fixed index, k=8) ==
  looped  x3: 67896 pairs, stable: True
  batched x3: 67896 pairs, stable: True
  identical: True   symmetric difference: 0
```

Given the same index, the two probes agree exactly and both are deterministic. So the batching is
equivalent, and the drift belongs to index construction — worth knowing if you ever diff two graph
builds and expect them to match.

`verify_resolve` reports a **partition** hash rather than the id→canon map, for the same reason:
`_uf_union` breaks rank ties by arrival order, so which member of a merged group ends up canonical
is not stable even when the grouping is.


### 6. Sharding: one database per profile

Two separate questions, and they have opposite answers.

**Ingest.** Sharding helped on 0.1.15, but only because it divided the quadratic — K shards turn one
O(N²) into K independent O((N/K)²). Measured over 400 documents *before* the fixes:

```
                    per-doc rebuild    deferred rebuild
1 shard                  111.82s            19.76s
4 shards                  33.19s            17.43s
8 shards                  23.76s            17.53s
```

8 shards bought 4.7x unfixed, and ~12% once fixed — and **the fix alone on one database (19.8s)
beat 8-way sharding without it (23.8s)**. So sharding is not an ingest-throughput tool any more.


In [ ]:
#| eval: false
from evals.ingest_bench import bench_shards, bench_shard_reads
bench_shards(n=400)
bench_shard_reads(n=1200, shards=(1,2,4,8,16))

**Reads, which is the part that actually decides the design:**

```
1200 docs, 60 queries x3, best-of-3, ms/query

 shards  chunks/shard   fanout   fanout+threads   routed
      1        25,041     8.04ms          8.00ms   8.25ms
      2        12,391     9.28ms         11.51ms   5.34ms
      4         6,216    13.20ms         14.68ms   3.22ms
      8         3,099    20.28ms         30.44ms   2.34ms
     16         1,608    34.85ms         52.56ms   2.06ms
```

- **Fan-out to every shard: 4.3x more expensive at 16 shards.** You pay K searches plus a fusion,
  and because HNSW is O(log N), K searches over N/K rows cost strictly more than one search over N.
  There is no shard count at which splitting the index reduces total work.
- **Routed to the shard holding the profile: 4.0x cheaper at 16 shards**, and it keeps improving,
  because the one search is over an index 1/K the size.
- **Threading the fan-out does not rescue it** — 0.65x at 16 shards, with a persistent pool, not one
  created per query. Per-query fixed cost multiplies by K faster than the threads overlap.

So the question is never "how many shards" but **"does the query know which shard"**. If a profile
is a filter the caller already has, shard freely and reads get faster. If queries have to search
everything, each shard is a tax on every query, and the reasons to shard are the ones that have
nothing to do with speed: bounding the resident HNSW index (usearch keeps vectors in memory, so
~1M x 512 dims of float16 is ~1 GB plus graph overhead), tenant isolation, independent re-ingest,
and cross-process parallelism.


### What is still open at 10⁶

- **`build_graph`** — ~40 chunks/s, single-core, accumulating in memory for the whole call.
- **Embedding** — deliberately excluded from every number here. With litesearch's own overhead out
  of the way, the encoder is now the dominant cost, and it is the part that wants a GPU or a
  process pool.
- **`add_dir` walks files serially** — parsing parallelises 1.9x on 4 cores and does not do so yet
  on the document path.
- **`resolve_entities` at exponent 1.08** — near-linear, not linear. `_lexical_pairs` still skips
  token blocks larger than `max_group`, so resolution quality degrades discontinuously as the
  corpus grows. Not a speed problem; a behaviour one, and unmeasured.
